<a href="https://colab.research.google.com/github/rypriyanka2005/Machine-Learning-Sem-4/blob/MLlabs/mlLab7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

output for other embidings

In [31]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    confusion_matrix
)

from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score
)

# Optional models
try:
    from xgboost import XGBClassifier
    xgb_available = True
except:
    xgb_available = False

try:
    from catboost import CatBoostClassifier
    catboost_available = True
except:
    catboost_available = False


# ================================
# 1. Load Data
# ================================
def load_data(file_path):
    data = np.load(file_path)

    X = data['features']
    y = data['labels']

    print(f"\n📂 Loading: {file_path}")
    print("Shape:", X.shape, "| Labels:", np.unique(y))

    return X, y


# ================================
# 2. Preprocess
# ================================
def preprocess_data(X, y):
    if y.dtype == object:
        y = LabelEncoder().fit_transform(y)

    scaler = StandardScaler()
    X = scaler.fit_transform(X)

    return X, y


# ================================
# 3. Split
# ================================
def split_data(X, y):
    return train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )


# ================================
# 4. Models
# ================================
def get_models():
    models = {
        "SVM": SVC(),
        "Decision Tree": DecisionTreeClassifier(),
        "Random Forest": RandomForestClassifier(n_estimators=100),
        "AdaBoost": AdaBoostClassifier(n_estimators=100),
        "Naive Bayes": GaussianNB(),
        "MLP": MLPClassifier(max_iter=300)
    }

    if xgb_available:
        models["XGBoost"] = XGBClassifier(
            eval_metric='logloss',
            n_estimators=100,
            verbosity=0
        )

    if catboost_available:
        models["CatBoost"] = CatBoostClassifier(
            verbose=0,
            iterations=100
        )

    return models


# ================================
# 5. Train & Evaluate
# ================================
def evaluate_models(X_train, X_test, y_train, y_test):

    models = get_models()
    results = []
    confusion_matrices = {}

    for name, model in models.items():
        print(f"🚀 Training {name}...")

        model.fit(X_train, y_train)

        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

        # Confusion matrices
        cm_train = confusion_matrix(y_train, y_train_pred)
        cm_test = confusion_matrix(y_test, y_test_pred)

        confusion_matrices[name] = {
            "train_cm": cm_train,
            "test_cm": cm_test
        }

        results.append({
            "Model": name,

            # Train metrics
            "Train Acc": accuracy_score(y_train, y_train_pred),
            "Train Precision": precision_score(y_train, y_train_pred, average='weighted', zero_division=0),
            "Train Recall": recall_score(y_train, y_train_pred, average='weighted', zero_division=0),
            "Train F1": f1_score(y_train, y_train_pred, average='weighted', zero_division=0),

            # Test metrics
            "Test Acc": accuracy_score(y_test, y_test_pred),
            "Test Precision": precision_score(y_test, y_test_pred, average='weighted', zero_division=0),
            "Test Recall": recall_score(y_test, y_test_pred, average='weighted', zero_division=0),
            "Test F1": f1_score(y_test, y_test_pred, average='weighted', zero_division=0)
        })

    return pd.DataFrame(results), confusion_matrices


# ================================
# 6. KMeans
# ================================
def run_kmeans(X_train, X_test, y_train, y_test):

    n_clusters = len(np.unique(y_train))

    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=42,
        n_init=10
    )

    train_clusters = kmeans.fit_predict(X_train)
    test_clusters = kmeans.predict(X_test)

    return pd.DataFrame([{
        "Model": "KMeans",
        "Train ARI": adjusted_rand_score(y_train, train_clusters),
        "Test ARI": adjusted_rand_score(y_test, test_clusters),
        "Train NMI": normalized_mutual_info_score(y_train, train_clusters),
        "Test NMI": normalized_mutual_info_score(y_test, test_clusters)
    }])


# ================================
# 7. Pipeline
# ================================
def run_pipeline(file_path):

    X, y = load_data(file_path)
    X, y = preprocess_data(X, y)

    if len(np.unique(y)) < 2:
        print("❌ Not enough classes")
        return

    X_train, X_test, y_train, y_test = split_data(X, y)

    results_df, confusion_matrices = evaluate_models(X_train, X_test, y_train, y_test)
    kmeans_df = run_kmeans(X_train, X_test, y_train, y_test)

    print("\n📊 Supervised:\n", results_df)
    print("\n📊 KMeans:\n", kmeans_df)

    # Print confusion matrices
    for model_name, cms in confusion_matrices.items():
        print(f"\n📌 Confusion Matrix - {model_name}")
        print("Train:\n", cms["train_cm"])
        print("Test:\n", cms["test_cm"])

    # Save results to Excel
    output_file = file_path.replace(".npz", "_results.xlsx")

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        results_df.to_excel(writer, sheet_name="Supervised", index=False)
        kmeans_df.to_excel(writer, sheet_name="KMeans", index=False)

        # Save confusion matrices
        for model_name, cms in confusion_matrices.items():
            pd.DataFrame(cms["train_cm"]).to_excel(writer, sheet_name=f"{model_name}_Train_CM")
            pd.DataFrame(cms["test_cm"]).to_excel(writer, sheet_name=f"{model_name}_Test_CM")

    print(f"\n✅ Saved: {output_file}")


# ================================
# 8. Main
# ==============  ==================
if __name__ == "__main__":

    files = [
        "vgg16_embeddingsfinal.npz",
        "vgg19_embeddingsfinal.npz",
        "resnet50_embeddingsfinal.npz"

    ]

    for file in files:
        run_pipeline(file)


📂 Loading: vgg16_embeddingsfinal.npz
Shape: (633, 512) | Labels: [0 1 2 3 4 5 6 7 8 9]
🚀 Training SVM...
🚀 Training Decision Tree...
🚀 Training Random Forest...
🚀 Training AdaBoost...
🚀 Training Naive Bayes...
🚀 Training MLP...
🚀 Training XGBoost...

📊 Supervised:
            Model  Train Acc  Train Precision  Train Recall  Train F1  \
0            SVM   0.747036         0.854270      0.747036  0.740181   
1  Decision Tree   0.994071         0.994600      0.994071  0.994184   
2  Random Forest   0.994071         0.994165      0.994071  0.994083   
3       AdaBoost   0.355731         0.357597      0.355731  0.296013   
4    Naive Bayes   0.679842         0.719408      0.679842  0.672684   
5            MLP   0.994071         0.994106      0.994071  0.994077   
6        XGBoost   0.994071         0.994118      0.994071  0.993931   

   Test Acc  Test Precision  Test Recall   Test F1  
0  0.370079        0.432373     0.370079  0.256826  
1  0.204724        0.217356     0.204724  0.206068

In [30]:
import numpy as np
from sklearn.preprocessing import LabelEncoder
import joblib

def encode_npz_labels(input_npz, output_npz, encoder_path=None):
    data = np.load(input_npz, allow_pickle=True)

    y = data['y']

    le = LabelEncoder()
    y_encoded = le.fit_transform(y)

    np.savez(output_npz, features=data['X'], labels=y_encoded)

    if encoder_path:
        joblib.dump(le, encoder_path)

    print("Classes mapping:")
    for i, cls in enumerate(le.classes_):
        print(f"{cls} → {i}")

    return le


# ✅ CALL FUNCTION HERE (outside)
encode_npz_labels(
    input_npz="resnet50_embeddings.npz",
    output_npz="resnet50_embeddingsfinal.npz",
    encoder_path="label_encoder.pkl"
)

Classes mapping:
0 → 0
1 → 1
2 → 2
3 → 3
4 → 4
5 → 5
6 → 6
7 → 7
8 → 8
9 → 9


LabelEncoder()

In [28]:
import numpy as np

data = np.load("resnet50_embeddings.npz", allow_pickle=True)
print("Keys inside npz file:", data.files)

Keys inside npz file: ['X', 'y']
